# 02 — Elo ratings and features
Tuning results, rating trajectories, and the leakage check.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))   # run from notebooks/
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pitchiq import config
pd.set_option("display.width", 160)
print("Reading from:", config.ROOT, "| synthetic:", config.SYNTHETIC)

In [ ]:
pd.read_csv(config.REPORTS / "tuning_elo.csv").head(10)

In [ ]:
import json
from pitchiq.ratings.elo import EloParams, run_elo
p = EloParams(**json.load(open(config.ARTIFACTS / "tuned_params.json"))["elo"])
m = pd.read_parquet(config.PROCESSED / "matches.parquet")
elo = run_elo(m, p).merge(m[["match_id","date","home","away"]], on="match_id")
long = pd.concat([elo[["date","home","elo_home"]].set_axis(["date","team","elo"], axis=1),
                  elo[["date","away","elo_away"]].set_axis(["date","team","elo"], axis=1)])
top = long.groupby("team").elo.last().nlargest(6).index
ax = long[long.team.isin(top)].pivot_table(index="date", columns="team", values="elo").ffill().plot(figsize=(11,4))
ax.set_title("Pre-match Elo, current top 6"); plt.show()

In [ ]:
f = pd.read_parquet(config.PROCESSED / "features.parquet")
from pitchiq.features.build import FEATURES
f[f.season >= 2015][FEATURES].describe().T.round(3)

In [ ]:
pd.read_csv(config.REPORTS / "leakage_check.csv")

Every tested date should show `leaked_features = none` **and** `later_features_changed = True`. The second column proves the scramble really changed later data, so the test isn't trivially passing.